Cell 1: Establish path and environment variables, including adding utils folder to system path for imports.  Also set up logging infrastructure for Audit Compliance.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
import os
import sys
import logging
from datetime import datetime
from pathlib import Path
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

# 1. Logging Infrastructure Configuration for Audit Compliance
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(filename)s:%(lineno)d | %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('notebook_2_cox_engine_execution.log', mode='w')
    ]
)
logger = logging.getLogger("CoxPH_Engine_Template")

# Dynamically locate data warehouse ROOT directory
notebook_path = Path(os.getcwd())
ROOT_DIR = notebook_path
while ROOT_DIR.name != "data_warehouse" and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DB_DIR = ROOT_DIR / "databases"
TRANSITORY_DB_PATH = DB_DIR / "transitory" / "peri_urban_ag_analysis.db"
MASTER_DB_PATH = DB_DIR / "sba_7a_analysis.db"
IRS_DB_PATH = DB_DIR / "irs_county_soi.db"

# Add the ROOT path to Python sys path for imports
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# Import custom classes now that syspath is defined.    
from utils.geography_daemon import GeographyDaemon
from utils.macro_feature_engine import MacroFeatureEngine



In [2]:
# =====================================================================
# Cell 2: High-Velocity Data Layer Loading & Core Isolate
# =====================================================================

logger.info(f"Establishing read-only connection to Transitory DB: {TRANSITORY_DB_PATH}")

if not TRANSITORY_DB_PATH.exists():
    logger.error(f"Execution halted: Target database not found at {TRANSITORY_DB_PATH}")
    raise FileNotFoundError(f"Missing analytical layer: {TRANSITORY_DB_PATH}")

conn = sqlite3.connect(f"file:{TRANSITORY_DB_PATH}?mode=ro", uri=True)
try:
    # FIXED: Direct SQL filtering isolates your pristine core target portfolio 
    # and leaves the background proxy rows safely on disk until called upon
    query = "SELECT * FROM source_loans_snapshot WHERE is_core_sample = 1"
    logger.info("Executing micro-data core portfolio snapshot pull into RAM...")
    df_cox = pd.read_sql_query(query, conn)
    logger.info(f"Successfully ingested {len(df_cox):,} core target records for survival analysis.")
finally:
    conn.close()

# Schema Integrity Audit & Data Typing
df_cox['survival_months'] = pd.to_numeric(df_cox['survival_months'], errors='coerce')
df_cox['event_occurred'] = pd.to_numeric(df_cox['event_occurred'], errors='coerce').astype(int)
df_cox['terminmonths'] = pd.to_numeric(df_cox['terminmonths'], errors='coerce').astype(int)
df_cox['is_long_duration'] = pd.to_numeric(df_cox['is_long_duration'], errors='coerce').astype(int)

df_cox = df_cox.dropna(subset=['survival_months', 'event_occurred', 'naics_4d', 'terminmonths'])


2026-06-25 18:00:46,414 | INFO | 1797106673.py:5 | Establishing read-only connection to Transitory DB: /Users/bonwier/PythonProjects/data_warehouse/databases/transitory/peri_urban_ag_analysis.db
2026-06-25 18:00:46,416 | INFO | 1797106673.py:16 | Executing micro-data core portfolio snapshot pull into RAM...
2026-06-25 18:00:46,826 | INFO | 1797106673.py:18 | Successfully ingested 45,651 core target records for survival analysis.


In [3]:
# =====================================================================
# Cell 3: Comprehensive Structural, Sector, & FIPS-State Diagnostic Scan
# =====================================================================
import pandas as pd
import numpy as np
from lifelines.statistics import logrank_test, multivariate_logrank_test

logger.info("Initializing Comprehensive Internal Variance Scan via FIPS Slicing...")

# 1. VARIABLE DIMENSION 1: Temporal Term Structure (Duration)
logger.info("Executing Log-Rank Test across asset duration splits...")
short_term_loans = df_cox[df_cox['is_long_duration'] == 0]
long_term_loans = df_cox[df_cox['is_long_duration'] == 1]

duration_p = 1.0
if len(short_term_loans) > 0 and len(long_term_loans) > 0:
    duration_test = logrank_test(
        durations_A=short_term_loans['survival_months'],
        durations_B=long_term_loans['survival_months'],
        event_observed_A=short_term_loans['event_occurred'],
        event_observed_B=long_term_loans['event_occurred']
    )
    duration_p = duration_test.p_value

# 2. VARIABLE DIMENSION 2: Sector Taxonomy (Industry)
logger.info("Executing Multivariate Log-Rank Test across all 4-Digit NAICS sectors...")
industry_p = 1.0
try:
    industry_test = multivariate_logrank_test(
        df_cox['survival_months'],
        df_cox['naics_4d'],
        df_cox['event_occurred']
    )
    industry_p = industry_test.p_value
except Exception as e:
    logger.warning(f" • Industry Test failed to converge: {str(e)}")

# 3. VARIABLE DIMENSION 3: Macro-Geography (State FIPS Prefix Slicing)
logger.info("Extracting 2-digit State FIPS prefixes from spatial tracking tokens...")
# Ensure standardized_fips is handled uniformly as a zero-padded string
df_cox['state_fips'] = df_cox['standardized_fips'].fillna('99').astype(str).str.strip().str.zfill(5).str[:2]

# Filter out unknown placeholders ('99') and states with fewer than 50 loans to preserve high data density
state_counts = df_cox['state_fips'].value_counts()
dense_states = state_counts[(state_counts >= 50) & (state_counts.index != '99')].index
df_state_dense = df_cox[df_cox['state_fips'].isin(dense_states)]

logger.info(f"Executing Multivariate Log-Rank Test across {df_state_dense['state_fips'].nunique()} State FIPS Regions...")
state_p = 1.0
try:
    state_test = multivariate_logrank_test(
        df_state_dense['survival_months'],
        df_state_dense['state_fips'],
        df_state_dense['event_occurred']
    )
    state_p = state_test.p_value
except Exception as e:
    logger.warning(f" • State Geographic Test failed to converge: {str(e)}")

# 4. State Default Rate Spread Analysis for Validation
state_stats = df_cox.groupby('state_fips').agg(
    total_funded=('event_occurred', 'count'),
    total_defaults=('event_occurred', 'sum')
).loc[dense_states]
state_stats['default_rate'] = (state_stats['total_defaults'] / state_stats['total_funded']) * 100

state_variance = state_stats['default_rate'].var()
state_max = state_stats['default_rate'].max()
state_min = state_stats['default_rate'].min()

print("\n=== CORESYNC COMPREHENSIVE NATIVE DIAGNOSTIC MATRIX ===")
print(f" • 1. Structural Duration P-Value:      {duration_p:.4e}")
print(f" • 2. 4-Digit Sector Taxonomy P-Value: {industry_p:.4e}")
print(f" • 3. State FIPS Geography P-Value:    {state_p:.4e}")
print("------------------------------------------------")
print(f" • Dense State Jurisdictions Analyzed:   {len(state_stats)}")
print(f" • State-Level Default Rate Variance:    {state_variance:.4f}")
print(f" • Max State Risk Baseline FIPS prefix:  {state_stats['default_rate'].idxmax()} ({state_max:.2f}%)")
print(f" • Min State Risk Baseline FIPS prefix:  {state_stats['default_rate'].idxmin()} ({state_min:.2f}%)")

# 5. STRATEGIC INSIGHT ROADMAP GENERATION
print("\n=== SYSTEM AUTOMATED DATA ROADMAP ===")
if state_p <= 0.005:
    print(f" --> [GEOGRAPHY PROVEN] State jurisdictions exhibit distinct survival profiles (Spread: {state_max - state_min:.2f}%).")
    print("     Developing regional baseline multipliers or macro-conditioning is heavily justified.")
else:
    print(" --> [GEOGRAPHY INSULATED] State boundaries show uniform baseline survival profiles.")


2026-06-25 18:01:24,237 | INFO | 2493867177.py:8 | Initializing Comprehensive Internal Variance Scan via FIPS Slicing...
2026-06-25 18:01:24,238 | INFO | 2493867177.py:11 | Executing Log-Rank Test across asset duration splits...
2026-06-25 18:01:24,292 | INFO | 2493867177.py:26 | Executing Multivariate Log-Rank Test across all 4-Digit NAICS sectors...
2026-06-25 18:01:24,578 | INFO | 2493867177.py:39 | Extracting 2-digit State FIPS prefixes from spatial tracking tokens...
2026-06-25 18:01:24,625 | INFO | 2493867177.py:48 | Executing Multivariate Log-Rank Test across 52 State FIPS Regions...

=== CORESYNC COMPREHENSIVE NATIVE DIAGNOSTIC MATRIX ===
 • 1. Structural Duration P-Value:      0.0000e+00
 • 2. 4-Digit Sector Taxonomy P-Value: 1.6353e-198
 • 3. State FIPS Geography P-Value:    1.1316e-114
------------------------------------------------
 • Dense State Jurisdictions Analyzed:   52
 • State-Level Default Rate Variance:    9.7160
 • Max State Risk Baseline FIPS prefix:  11 (16.00%

Martingale Residuals Scan to determine linearity of loan duration on default risk.

In [7]:
# =====================================================================
# Cell 4: Continuous Endogenous Maturity Non-Linearity Diagnostic
# =====================================================================
import sqlite3
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter

logger.info("--- Executing Matrix-Insulated Endogenous Maturity Scan ---")

# 1. HARD RESET: Rebuild the data frame from scratch
conn = sqlite3.connect(TRANSITORY_DB_PATH)
try:
    query = "SELECT survival_months, event_occurred, terminmonths FROM source_loans_snapshot WHERE is_core_sample = 1"
    df_isolated_matrix = pd.read_sql_query(query, conn)
finally:
    conn.close()

# 2. Schema Type Enforcements
df_isolated_matrix['survival_months'] = pd.to_numeric(df_isolated_matrix['survival_months'], errors='coerce')
df_isolated_matrix['event_occurred'] = pd.to_numeric(df_isolated_matrix['event_occurred'], errors='coerce').astype(int)
df_isolated_matrix['terminmonths'] = pd.to_numeric(df_isolated_matrix['terminmonths'], errors='coerce').astype(int)

df_isolated_matrix = df_isolated_matrix.dropna().reset_index(drop=True)
logger.info(f" • Pristine firewalled modeling space locked at: {len(df_isolated_matrix):,} records.")

# 3. Fit the Clean Baseline Test Model
cph_check = CoxPHFitter(penalizer=0.0)
cph_check.fit(
    df_isolated_matrix,
    duration_col='survival_months',
    event_col='event_occurred',
    show_progress=False
)

# 4. FIXED: Row-by-Row Expected Cumulative Hazard Vector Lookup
# Extract the unique baseline cumulative hazard function
baseline_cum_hazard = cph_check.baseline_cumulative_hazard_

# Step-adjust: Find the baseline hazard value corresponding to each loan's specific exit month
# We use np.minimum to safely bound any floating tracking durations to your baseline curve index bounds
max_baseline_time = baseline_cum_hazard.index.max()
bounded_times = np.minimum(df_isolated_matrix['survival_months'].values, max_baseline_time)
H_0_t = baseline_cum_hazard.loc[bounded_times].values.flatten()

# Extract the partial hazard multiplier per individual loan risk profile: exp(beta * X)
partial_hazards = cph_check.predict_partial_hazard(df_isolated_matrix).to_numpy().flatten()

# Exact Expected Value vector = Baseline Hazard at Exit Time * Partial Hazard Risk Multiplier
expected_values = H_0_t * partial_hazards

# Calculate pure Martingale Residuals: Actual Event (0 or 1) - Expected Value Vector
# This is now a clean 1D subtraction matching your 45,651 rows perfectly
df_isolated_matrix['residuals'] = df_isolated_matrix['event_occurred'].values - expected_values

# 5. Group by 12-month increments to map the error trend lines
df_isolated_matrix['term_bucket'] = (df_isolated_matrix['terminmonths'] // 12) * 12
term_profile = df_isolated_matrix.groupby('term_bucket').agg(
    total_loans=('event_occurred', 'count'),
    observed_defaults=('event_occurred', 'sum'),
    mean_residual=('residuals', 'mean')
).reset_index()

print("\n=== CORESYNC ENDOGENOUS PROFILE: MATURITY TIER ANALYSIS ===")
df_filtered_profile = term_profile[term_profile['total_loans'] >= 50].copy()
print(df_filtered_profile.to_string(index=False, formatters={
    'mean_residual': '{:,.4f}'.format
}))

# 6. Automated Structural Recommendation Logic
residuals_clean = df_filtered_profile['mean_residual'].values
direction_changes = np.diff(np.sign(residuals_clean))

print("\n=== SYSTEM ARCHITECTURE RECOMMENDATION ===")
if np.any(direction_changes != 0):
    print(" --> [CRITICAL INFLECTION DETECTED] Residual signs flip across maturity groups.")
    print("     Maturity risk is highly non-linear. Pathway 3 (Discrete Dividing Tiers) is mandatory.")
else:
    print(" --> [SMOOTH PROFILE] Residuals scale linearly. Pathway 2 (Continuous HRMs) is acceptable.")



2026-06-25 18:16:15,670 | INFO | 509278629.py:9 | --- Executing Matrix-Insulated Endogenous Maturity Scan ---
2026-06-25 18:16:15,765 | INFO | 509278629.py:25 |  • Pristine firewalled modeling space locked at: 45,651 records.

=== CORESYNC ENDOGENOUS PROFILE: MATURITY TIER ANALYSIS ===
 term_bucket  total_loans  observed_defaults mean_residual
           0          640                147       -0.3253
          12         1545                134       -0.0932
          24          979                363        0.1276
          36         1529                506        0.1780
          48         1233                476        0.2653
          60         4770                512        0.0011
          72         1225                236        0.1021
          84        10258                220       -0.0483
          96          826                208        0.2049
         108          733                 68        0.0568
         120        12410                 33       -0.0199
     